# Generalized MRVI-based SCHEMATIC analysis notebook

This notebook is a reusable template for running an MRVI-based single-cell transcriptomic classification of inhibitors. Configure paths and metadata columns once, then run the workflow end-to-end or resume from saved intermediate outputs.

**Expected inputs**
- An AnnData `.h5ad` file, **or** a Matrix Market counts file plus gene metadata and cell metadata.
- Cell metadata columns identifying biological group/sample, batch/replicate, and optionally labels/covariates.

**Main outputs**
- Trained MRVI model
- Latent representations (`u` and `z`)
- Cell-level and sample-level counterfactual distances
- Drug/Drug-dose/condition response modules
- Multivariate differential expression of response modules
- Transcriptional drug classification based on response module cluster membership

## 1. Optional environment setup
Run this cell only when packages are missing. In managed environments, prefer installing dependencies outside the notebook.

In [ ]:
# Uncomment if needed.
# %pip install scanpy anndata scvi-tools xarray seaborn scipy scikit-learn
# %pip install --quiet scvi-colab
# from scvi_colab import install
# install()

## 2. Imports and global settings

In [ ]:
from pathlib import Path
import json
import pickle
import warnings

import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import xarray as xr
import scipy.sparse as sp
from scipy.cluster import hierarchy
from scipy.spatial import distance
from scipy.cluster.hierarchy import fcluster
from sklearn.metrics import silhouette_score, silhouette_samples

import matplotlib.pyplot as plt
import seaborn as sns

import scvi
from scvi.external import MRVI
import flax.linen as nn

warnings.filterwarnings("ignore")
scvi.settings.seed = 0
sns.set_theme(style="whitegrid")
print(f"scanpy={sc.__version__}, scvi-tools={scvi.__version__}")

## 3. Configuration
Edit this section for a new dataset. The rest of the notebook references `CONFIG` rather than hard-coded paths or column names.

In [ ]:
CONFIG = {
    # Project naming
    "project_name": "example_project",
    "output_dir": "outputs/example_project",

    # Input mode: choose "h5ad" or "mtx"
    "input_mode": "h5ad",
    "h5ad_path": "path/to/input.h5ad",

    # Used only when input_mode == "mtx"
    "matrix_path": "path/to/counts.mtx",
    "genes_path": "path/to/genes.tsv",
    "cell_metadata_path": "path/to/cell_metadata.csv",
    "gene_name_column": 0,         # integer column index or string column name
    "cell_metadata_index_col": 0,  # set to None if metadata has no index column

    # Metadata columns in adata.obs
    "batch_key": "replicate",
    "sample_key": "sample",       # e.g. a pre-existing drug+dose column
    "labels_key": None,            # optional; set to a column name or None

    # Optional: create sample_key from multiple obs columns
    "create_sample_key": True,
    "sample_key_parts": ["drug_name", "drug_dose"],
    "sample_key_sep": "_",

    # Optional filtering/subsetting. Leave values as None to skip.
    "subset_query": None,          # e.g. "cell_line == 'BT112'"
    "hvg_groupby_columns": ["cell_line", "drug_name"],
    "n_top_genes_per_group": 100,
    "min_gene_cell_fraction": 0.05,

    # Model settings
    "model_dir": "mrvi_model",
    "load_existing_model": False,
    "max_epochs": 100,
    "batch_size": 256,
    "train_size": 0.9,
    "early_stopping_patience": 15,

    # Distance / clustering / DE settings
    # Keep False for memory efficiency. Cell-level distances are computed in chunks below.
    "keep_cell_distances": False,
    "cell_distance_chunk_size": 2000,
    "distance_cluster_method": "complete",
    "distance_cluster_threshold": None,  # if None, choose via maxclust
    "n_response_clusters": 4,
    "control_sample_regex": "DMSO|control|vehicle",
    "lfc_threshold": 0.2,
    "top_n_genes": 100,

    # Final transcriptional drug-class settings
    # drug_name_key is used to collapse multiple drug+dose samples back to a drug-level class.
    # If it is missing from adata.obs, the notebook falls back to parsing sample_key values.
    "drug_name_key": "drug_name",
    "drug_class_cluster_method": "complete",
    "drug_class_distance_threshold": 0.5,
}

OUT = Path(CONFIG["output_dir"])
OUT.mkdir(parents=True, exist_ok=True)
with open(OUT / "config.json", "w") as f:
    json.dump(CONFIG, f, indent=2)
OUT

## 4. Helper functions

In [ ]:
def require_columns(df: pd.DataFrame, columns, df_name="DataFrame"):
    missing = [c for c in columns if c is not None and c not in df.columns]
    if missing:
        raise ValueError(f"{df_name} is missing required columns: {missing}")


def make_sample_key(adata, key, parts, sep="_"):
    require_columns(adata.obs, parts, "adata.obs")
    adata.obs[key] = adata.obs[parts].astype(str).agg(sep.join, axis=1)
    return adata


def load_anndata(config):
    if config["input_mode"] == "h5ad":
        adata = sc.read_h5ad(config["h5ad_path"])
    elif config["input_mode"] == "mtx":
        adata = sc.read_mtx(config["matrix_path"]).T  # cells x genes is expected downstream
        genes = pd.read_csv(config["genes_path"], sep=None, engine="python", header=None)
        gene_col = config["gene_name_column"]
        adata.var_names = genes[gene_col].astype(str).values
        obs = pd.read_csv(config["cell_metadata_path"], index_col=config["cell_metadata_index_col"])
        if obs.shape[0] != adata.n_obs:
            raise ValueError(f"cell metadata rows ({obs.shape[0]}) != matrix cells ({adata.n_obs})")
        adata.obs = obs.copy()
    else:
        raise ValueError("input_mode must be 'h5ad' or 'mtx'")
    adata.var_names_make_unique()
    return adata


def select_features_by_group_hvg(adata, groupby_columns, n_top_genes=100, min_gene_cell_fraction=0.05):
    """Union HVGs across metadata groups, then filter low-prevalence genes."""
    require_columns(adata.obs, groupby_columns, "adata.obs")
    hvg_union = set()
    groups = adata.obs[groupby_columns].drop_duplicates()
    for _, row in groups.iterrows():
        mask = np.ones(adata.n_obs, dtype=bool)
        for col in groupby_columns:
            mask &= adata.obs[col].astype(str).values == str(row[col])
        if mask.sum() < 3:
            continue
        tmp = adata[mask].copy()
        sc.pp.log1p(tmp)
        sc.pp.highly_variable_genes(
            tmp, flavor="seurat", n_top_genes=min(n_top_genes, tmp.n_vars), inplace=True
        )
        hvg_union |= set(tmp.var_names[tmp.var["highly_variable"]])
    if not hvg_union:
        raise ValueError("No HVGs selected. Check hvg_groupby_columns or group sizes.")
    adata = adata[:, sorted(hvg_union)].copy()
    min_cells = max(1, int(min_gene_cell_fraction * adata.n_obs))
    sc.pp.filter_genes(adata, min_cells=min_cells)
    return adata


def flatten_cell_sample_distances(cell_normalized_dists, adata, sample_key):
    dist_vec, sample_anno = [], []
    sample_names = cell_normalized_dists["sample_x"].to_numpy()
    for i in range(cell_normalized_dists["cell"].shape[0]):
        sample = adata.obs[sample_key].iloc[i]
        sample_idx = np.where(sample_names == sample)[0][0]
        vec = cell_normalized_dists["cell"][i, sample_idx, :].to_numpy()
        dist_vec.append(vec)
        sample_anno.append([sample] * len(vec))
    return np.concatenate(dist_vec), np.concatenate(sample_anno)

def _dataset_has_cell_distances(distance_ds):
    """Return True when a get_local_sample_distances result includes per-cell distances."""
    return "cell" in getattr(distance_ds, "data_vars", {})


def compute_cell_distances_in_chunks(model, adata, config, out_dir):
    """
    Compute memory-heavy per-cell local sample distances in chunks.

    This keeps `keep_cell_distances=False` for the main model workflow, then temporarily
    requests `keep_cell=True` on small AnnData slices only for downstream summaries that
    explicitly need cell-level distances.
    """
    chunk_size = int(config.get("cell_distance_chunk_size", 2000))
    if chunk_size <= 0:
        raise ValueError("cell_distance_chunk_size must be a positive integer")

    out_dir = Path(out_dir)
    chunk_dir = out_dir / "cell_distance_chunks"
    chunk_dir.mkdir(parents=True, exist_ok=True)

    sample_names = None
    sample_distance_sum = None
    n_cells_seen = 0
    chunk_paths = []

    for start in range(0, adata.n_obs, chunk_size):
        stop = min(start + chunk_size, adata.n_obs)
        adata_chunk = adata[start:stop].copy()

        chunk_dists = model.get_local_sample_distances(
            adata_chunk,
            use_mean=True,
            normalize_distances=True,
            keep_cell=True,
            groupby=config["labels_key"],
        )
        if not _dataset_has_cell_distances(chunk_dists):
            raise RuntimeError("Expected per-cell distances in chunk result, but none were returned.")

        cell_da = chunk_dists["cell"].load()
        if sample_names is None:
            sample_names = cell_da["sample_x"].values
            sample_distance_sum = np.zeros((len(sample_names), len(sample_names)), dtype=float)

        sample_distance_sum += cell_da.sum(axis=0).to_numpy()
        n_cells_seen += cell_da.shape[0]

        dist_vec, sample_anno = flatten_cell_sample_distances(
            chunk_dists,
            adata_chunk,
            config["sample_key"],
        )
        chunk_df = pd.DataFrame({
            "sample": sample_anno,
            "counterfactual_distance": dist_vec,
            "log_counterfactual_distance": np.log1p(dist_vec),
        })
        chunk_path = chunk_dir / f"counterfactual_distances_cells_{start:08d}_{stop:08d}.csv"
        chunk_df.to_csv(chunk_path, index=False)
        chunk_paths.append(chunk_path)

        # Drop chunk-local distance objects promptly to release memory in notebook sessions.
        del adata_chunk, chunk_dists, cell_da, chunk_df, dist_vec, sample_anno

    if n_cells_seen == 0:
        raise RuntimeError("No cells were processed while computing chunked distances.")

    sample_dist_df = pd.DataFrame(
        sample_distance_sum / n_cells_seen,
        columns=sample_names,
        index=sample_names,
    )
    return sample_dist_df, chunk_paths



## 5. Load and validate data

In [ ]:
adata = load_anndata(CONFIG)

if CONFIG["create_sample_key"]:
    make_sample_key(
        adata,
        key=CONFIG["sample_key"],
        parts=CONFIG["sample_key_parts"],
        sep=CONFIG["sample_key_sep"],
    )

required = [CONFIG["batch_key"], CONFIG["sample_key"], CONFIG["labels_key"]]
require_columns(adata.obs, required, "adata.obs")

if CONFIG["subset_query"]:
    keep = adata.obs.eval(CONFIG["subset_query"])
    adata = adata[keep].copy()

print(adata)
adata.obs[[CONFIG["batch_key"], CONFIG["sample_key"]]].head()

## 6. Feature selection

In [ ]:
adata = select_features_by_group_hvg(
    adata,
    groupby_columns=CONFIG["hvg_groupby_columns"],
    n_top_genes=CONFIG["n_top_genes_per_group"],
    min_gene_cell_fraction=CONFIG["min_gene_cell_fraction"],
)
adata.write_h5ad(OUT / f"{CONFIG['project_name']}_feature_selected.h5ad")
adata

## 7. Set up and train MRVI

In [ ]:
MRVI.setup_anndata(
    adata,
    batch_key=CONFIG["batch_key"],
    sample_key=CONFIG["sample_key"],
    labels_key=CONFIG["labels_key"],
)

model_kwargs = {
    "n_latent": 30,
    "n_latent_u": 10,
    "qz_nn_flavor": "attention",
    "px_nn_flavor": "attention",
    "qz_kwargs": {
        "use_map": True,
        "stop_gradients": False,
        "stop_gradients_mlp": True,
        "dropout_rate": 0.03,
    },
    "px_kwargs": {
        "stop_gradients": False,
        "stop_gradients_mlp": True,
        "h_activation": nn.softmax,
        "low_dim_batch": True,
        "dropout_rate": 0.03,
    },
    "learn_z_u_prior_scale": False,
    "z_u_prior": True,
    "u_prior_mixture": False,
    "u_prior_mixture_k": 20,
}

model_path = OUT / CONFIG["model_dir"]
if CONFIG["load_existing_model"] and model_path.exists():
    model = MRVI.load(model_path, adata=adata)
else:
    model = MRVI(adata, **model_kwargs)
    model.train(
        max_epochs=CONFIG["max_epochs"],
        batch_size=CONFIG["batch_size"],
        early_stopping=True,
        early_stopping_patience=CONFIG["early_stopping_patience"],
        check_val_every_n_epoch=1,
        train_size=CONFIG["train_size"],
        plan_kwargs={"lr": 2e-3, "n_epochs_kl_warmup": 5},
    )
    model.save(model_path, overwrite=True, save_anndata=False)
model

## 8. Latent representations and sample-level distances


In [ ]:
model_name = CONFIG["project_name"]
u_latent_key = f"X_{model_name}_u"
z_latent_key = f"X_{model_name}_z"

adata.obsm[u_latent_key] = model.get_latent_representation(adata, give_z=False)
adata.obsm[z_latent_key] = model.get_latent_representation(adata, give_z=True)
adata.uns["latent_keys"] = [u_latent_key, z_latent_key]

# Keep the main distance call memory-light. This object should contain sample-level
# distances/summaries, but not the full cell-level distance tensor.
sample_normalized_dists = model.get_local_sample_distances(
    adata,
    use_mean=True,
    normalize_distances=True,
    keep_cell=CONFIG["keep_cell_distances"],
    groupby=CONFIG["labels_key"],
)

if _dataset_has_cell_distances(sample_normalized_dists):
    raise RuntimeError(
        "CONFIG['keep_cell_distances'] is expected to be False here, but per-cell distances were retained."
    )

adata.write_h5ad(OUT / f"{CONFIG['project_name']}_mrvi_latents.h5ad")
sample_normalized_dists.to_netcdf(OUT / f"{CONFIG['project_name']}_sample_normalized_dists.nc")

pd.DataFrame(adata.obsm[u_latent_key], index=adata.obs_names).to_csv(OUT / f"{CONFIG['project_name']}_latent_u.csv")
pd.DataFrame(adata.obsm[z_latent_key], index=adata.obs_names).to_csv(OUT / f"{CONFIG['project_name']}_latent_z.csv")
sample_normalized_dists


## 9. Counterfactual distance summaries

In [ ]:
sample_dist_df, counterfactual_chunk_paths = compute_cell_distances_in_chunks(
    model=model,
    adata=adata,
    config=CONFIG,
    out_dir=OUT,
)
sample_dist_df.to_csv(OUT / f"{CONFIG['project_name']}_sample_distance_matrix.csv")

# Optional compact manifest instead of one very large in-memory DataFrame.
pd.DataFrame({"path": [str(p) for p in counterfactual_chunk_paths]}).to_csv(
    OUT / f"{CONFIG['project_name']}_counterfactual_distance_chunks.csv",
    index=False,
)

# Plot a histogram from chunk files without retaining all distances in memory.
plt.figure(figsize=(7, 4))
for path in counterfactual_chunk_paths:
    chunk = pd.read_csv(path, usecols=["counterfactual_distance"])
    plt.hist(chunk["counterfactual_distance"], bins=100, alpha=0.35)
plt.xlabel("Counterfactual distance")
plt.ylabel("Cell-sample comparisons")
plt.title("Distribution of cell-level counterfactual distances")
plt.show()

sample_dist_df.head()


## 10. Cluster samples into response classes

In [ ]:
cluster_method = CONFIG["distance_cluster_method"]
linkage = hierarchy.linkage(distance.pdist(sample_dist_df), method=cluster_method)

if CONFIG["distance_cluster_threshold"] is not None:
    cluster_labels = fcluster(linkage, t=CONFIG["distance_cluster_threshold"], criterion="distance")
else:
    cluster_labels = fcluster(linkage, t=CONFIG["n_response_clusters"], criterion="maxclust")

sample_cluster = pd.DataFrame({"cluster": cluster_labels}, index=sample_dist_df.index)
sample_cluster.to_csv(OUT / f"{CONFIG['project_name']}_sample_response_clusters.csv")

stability = silhouette_score(sample_dist_df.values, sample_cluster["cluster"].values, metric="euclidean")
print(f"Silhouette score: {stability:.3f}")

lut = dict(zip(sorted(sample_cluster["cluster"].unique()), sns.color_palette(n_colors=sample_cluster["cluster"].nunique())))
row_colors = sample_cluster["cluster"].map(lut)
sns.clustermap(
    sample_dist_df,
    row_linkage=linkage,
    col_linkage=linkage,
    row_colors=row_colors,
    col_colors=row_colors,
    cmap="viridis",
    figsize=(10, 10),
)
plt.show()

sample_cluster.head()

## 11. Differential expression by response module (current MrVI API)
This section follows the current scvi-tools MrVI pattern: create a sample-level categorical covariate for response modules, then call `model.differential_expression(sample_cov_keys=[...], store_lfc=True)`.

Control samples are identified by `control_sample_regex`, they are used as the baseline category.


In [ ]:
response_module_key = "response_module"
sample_key = CONFIG["sample_key"]

# Map each sample to its response module from the distance-based clustering step.
sample_to_module = sample_cluster["cluster"].astype(int).map(lambda x: f"cluster_{x}")
sample_to_module.index = sample_to_module.index.astype(str)

# Optionally label matched controls as the baseline group.
control_mask_by_sample = sample_to_module.index.to_series().str.contains(
    CONFIG["control_sample_regex"], case=False, regex=True, na=False
)
if control_mask_by_sample.any():
    sample_to_module.loc[control_mask_by_sample.values] = "control"
    response_module_categories = ["control"] + sorted(
        [x for x in sample_to_module.unique() if x != "control"],
        key=lambda x: int(x.split("_")[-1]),
    )
else:
    response_module_categories = sorted(
        sample_to_module.unique(), key=lambda x: int(x.split("_")[-1])
    )
    print(
        "No control samples matched control_sample_regex; "
        f"using {response_module_categories[0]!r} as the baseline category."
    )

# Store the response-module covariate on adata.obs for plotting and cell subsetting.
adata.obs[response_module_key] = adata.obs[sample_key].astype(str).map(sample_to_module)
adata.obs[response_module_key] = pd.Categorical(
    adata.obs[response_module_key], categories=response_module_categories, ordered=True
)

# Store the same sample-level covariate on model.sample_info, where MrVI expects sample covariates.
sample_info = model.sample_info.copy()
candidate_sample_cols = [sample_key, "sample_id", "_scvi_sample"]
sample_info_sample_col = next(
    (
        col for col in candidate_sample_cols
        if col in sample_info.columns
        and sample_info[col].astype(str).isin(sample_to_module.index).any()
    ),
    None,
)

if sample_info_sample_col is None:
    # Fallback: assume sample_info index corresponds to sample labels.
    sample_info["_sample_label_for_response_module"] = sample_info.index.astype(str)
    sample_info_sample_col = "_sample_label_for_response_module"

sample_info[response_module_key] = sample_info[sample_info_sample_col].astype(str).map(sample_to_module)
sample_info[response_module_key] = pd.Categorical(
    sample_info[response_module_key], categories=response_module_categories, ordered=True
)

if sample_info[response_module_key].isna().any():
    missing = sample_info.loc[sample_info[response_module_key].isna(), sample_info_sample_col].astype(str).unique()[:10]
    raise ValueError(
        f"Some samples in model.sample_info were not assigned a response module. Examples: {missing}"
    )

model.sample_info = sample_info
print(model.sample_info[[sample_info_sample_col, response_module_key]].drop_duplicates().head())
print("Response-module category order:", response_module_categories)


In [ ]:
# Current MrVI DE API: one call over sample-level covariates.
de_res = model.differential_expression(
    sample_cov_keys=[response_module_key],
    store_lfc=True,
)

with open(OUT / f"{CONFIG['project_name']}_response_module_de_result.pkl", "wb") as f:
    pickle.dump(de_res, f, pickle.HIGHEST_PROTOCOL)

# Coefficients are named like "response_module_cluster_2" and are interpreted
# relative to the first category in response_module_categories.
covariates = [
    str(c) for c in de_res.covariate.values
    if str(c).startswith(f"{response_module_key}_")
]
print("DE covariates:", covariates)

# Add cell-wise effect-size summaries to adata.obs for UMAP inspection.
for covariate in covariates:
    obs_key = f"{covariate}_DE_eff_size"
    adata.obs[obs_key] = de_res.effect_size.sel(covariate=covariate).values

adata.write_h5ad(OUT / f"{CONFIG['project_name']}_mrvi_with_response_module_de.h5ad")


## 12. Extract response-module DE signatures
For each response-module coefficient, this section averages MrVI LFCs across cells in that response module, ranks genes by absolute average LFC, and writes both full and top-gene tables.


In [ ]:
lfc_threshold = CONFIG["lfc_threshold"]
top_n = CONFIG["top_n_genes"]

response_module_lfc = pd.DataFrame(index=adata.var_names)
de_dfs = {}
full_dfs = {}

def covariate_to_module(covariate: str) -> str:
    return covariate.replace(f"{response_module_key}_", "", 1)

for covariate in covariates:
    module_name = covariate_to_module(covariate)
    module_cell_names = adata.obs_names[adata.obs[response_module_key].astype(str) == module_name]
    if len(module_cell_names) == 0:
        # This can happen for baseline/reference categories that do not receive a coefficient.
        module_cell_names = adata.obs_names

    avg = de_res.sel(cell_name=module_cell_names, covariate=covariate).mean(dim="cell_name")
    lfc = avg.lfc.to_pandas().astype(float)

    lfc_df = pd.DataFrame({
        "gene": lfc.index,
        "LFC": lfc.values,
        "absLFC": np.abs(lfc.values),
        "response_module": module_name,
        "covariate": covariate,
        "n_cells_averaged": len(module_cell_names),
    })

    # Include p-values or other available gene-level summaries if present in this scvi-tools version.
    for value_name in ["p_value", "pvalue", "proba_de", "bayes_factor", "effect_size"]:
        if value_name in avg.data_vars:
            values = avg[value_name].to_pandas()
            if values.ndim == 1 and len(values) == len(lfc_df):
                lfc_df[value_name] = values.values

    full_dfs[module_name] = lfc_df.sort_values("absLFC", ascending=False)
    de_dfs[module_name] = full_dfs[module_name].loc[
        full_dfs[module_name]["absLFC"] >= lfc_threshold
    ].copy()
    response_module_lfc[module_name] = lfc_df.set_index("gene")["LFC"]

response_module_lfc.to_csv(OUT / f"{CONFIG['project_name']}_response_module_lfc.csv")

signature_table = pd.DataFrame({
    module_name: de_df.head(top_n)["gene"].reset_index(drop=True)
    for module_name, de_df in de_dfs.items()
})
signature_table.to_csv(OUT / f"{CONFIG['project_name']}_top_response_module_de_signatures.csv", index=False)
signature_table.head()


## 13. Response-module DE heatmap


In [ ]:
de_genes = sorted(set().union(*[set(df["gene"]) for df in de_dfs.values() if not df.empty]))
if de_genes:
    response_deg_df = response_module_lfc.loc[de_genes].fillna(0)
    response_deg_df.to_csv(OUT / f"{CONFIG['project_name']}_response_module_de_gene_lfc_matrix.csv")
    sns.clustermap(response_deg_df, cmap="icefire", center=0, figsize=(10, 10))
    plt.show()
else:
    print("No DE genes passed the threshold. Consider lowering lfc_threshold.")


## 14. Final transcriptional drug classes

This section reproduces the original notebook's final drug-class step in a generalized way.

It collapses drug+dose-level response-module assignments into a **drug × response-module membership matrix**. Each row is a drug, each column is a response module, and a value of 1 means at least one dose of that drug was assigned to that response module. Drugs are then hierarchically clustered by that binary membership profile to define final transcriptional drug classes.

For the original manuscript, response modules were defined for 3 cell models separately and were combined by clustering after determining covariate-specific differential gene expression LFCs. Final transcriptional drug class (TDC) membership was based on clusters of response modules. In this example notebook of one cell model, TDCs are based on response module membership (without

In [ ]:
drug_name_key = CONFIG.get("drug_name_key", "drug_name")
sample_key = CONFIG["sample_key"]
module_col = "cluster"

if "sample_cluster" not in globals():
    sample_cluster = pd.read_csv(
        OUT / f"{CONFIG['project_name']}_sample_response_clusters.csv",
        index_col=0,
    )

sample_to_module_int = sample_cluster[module_col].astype(int)
response_modules = sorted(sample_to_module_int.unique())

# Build a sample -> drug map. Prefer the explicit drug_name_key in adata.obs; otherwise
# fall back to removing the trailing separator-delimited part from sample_key values
# (for example, drug_name_dose -> drug_name).
if drug_name_key in adata.obs.columns:
    sample_to_drug = (
        adata.obs[[sample_key, drug_name_key]]
        .dropna()
        .astype(str)
        .drop_duplicates()
        .groupby(sample_key)[drug_name_key]
        .agg(lambda x: x.value_counts().index[0])
    )
else:
    sep = CONFIG.get("sample_key_sep", "_")
    print(
        f"{drug_name_key!r} was not found in adata.obs; "
        f"parsing drug names from {sample_key!r} by dropping the final {sep!r}-delimited token."
    )
    sample_to_drug = pd.Series(
        {
            str(sample): sep.join(str(sample).split(sep)[:-1])
            if sep in str(sample) else str(sample)
            for sample in sample_to_module_int.index
        }
    )

sample_to_drug = sample_to_drug.reindex(sample_to_module_int.index.astype(str))
if sample_to_drug.isna().any():
    missing = sample_to_drug.index[sample_to_drug.isna()].tolist()[:10]
    raise ValueError(
        "Some sample-level response modules could not be mapped back to drugs. "
        f"Examples: {missing}. Check CONFIG['drug_name_key'] and CONFIG['sample_key']."
    )

# Binary membership: does this drug have any dose/sample assigned to this response module?
drug_class_membership_df = pd.DataFrame(
    0,
    index=sorted(sample_to_drug.unique()),
    columns=response_modules,
    dtype=int,
)

for sample, module in sample_to_module_int.items():
    drug = sample_to_drug.loc[str(sample)]
    drug_class_membership_df.loc[drug, int(module)] = 1

drug_class_membership_df.index.name = drug_name_key
drug_class_membership_df.columns.name = "response_module"
drug_class_membership_df.to_csv(
    OUT / f"{CONFIG['project_name']}_drug_by_response_module_membership.csv"
)

drug_class_membership_df.head()

In [ ]:
cluster_method = CONFIG.get("drug_class_cluster_method", "complete")
distance_threshold = CONFIG.get("drug_class_distance_threshold", 0.5)

if drug_class_membership_df.shape[0] < 2:
    raise ValueError("Need at least two drugs to cluster final transcriptional drug classes.")

row_linkage = hierarchy.linkage(
    distance.pdist(drug_class_membership_df.values),
    method=cluster_method,
)

transcriptional_class_labels = fcluster(
    row_linkage,
    t=distance_threshold,
    criterion="distance",
)

transcriptional_drug_classes = pd.DataFrame(
    {"transcriptional_drug_class": transcriptional_class_labels.astype(int)},
    index=drug_class_membership_df.index,
)
transcriptional_drug_classes.to_csv(
    OUT / f"{CONFIG['project_name']}_final_transcriptional_drug_classes.csv"
)

lut = dict(
    zip(
        sorted(transcriptional_drug_classes["transcriptional_drug_class"].unique()),
        sns.hls_palette(
            transcriptional_drug_classes["transcriptional_drug_class"].nunique(),
            l=0.5,
            s=0.8,
        ),
    )
)
row_colors = transcriptional_drug_classes["transcriptional_drug_class"].map(lut)

g = sns.clustermap(
    drug_class_membership_df,
    cmap="viridis",
    xticklabels=True,
    row_linkage=row_linkage,
    row_colors=row_colors,
    yticklabels=True,
    figsize=(15, 15),
)

for label in sorted(lut):
    g.ax_row_dendrogram.bar(0, 0, color=lut[label], label=label, linewidth=0)
g.ax_row_dendrogram.legend(
    title="Transcriptional drug class",
    loc="lower left",
    bbox_to_anchor=(-0.08, 0.818),
    bbox_transform=plt.gcf().transFigure,
)
plt.show()

transcriptional_drug_classes.head()

## 15. Inspect latent spaces with UMAP

In [ ]:
def plot_latent_umap(adata, latent_key, color=None, n_neighbors=50, min_dist=0.1):
    latent_adata = sc.AnnData(adata.obsm[latent_key], obs=adata.obs.copy(), uns=adata.uns.copy())
    sc.pp.pca(latent_adata, use_highly_variable=False)
    sc.pp.neighbors(latent_adata, n_neighbors=n_neighbors, n_pcs=0)
    sc.tl.umap(latent_adata, min_dist=min_dist)
    sc.pl.umap(latent_adata, color=color, ncols=1)
    return latent_adata

color_cols = [c for c in [CONFIG["sample_key"], CONFIG["batch_key"], CONFIG["labels_key"]] if c is not None]
adata_u = plot_latent_umap(adata, u_latent_key, color=color_cols)
adata_z = plot_latent_umap(adata, z_latent_key, color=color_cols)

## 16. Resume from saved outputs
Use this section to reload previously saved artifacts instead of rerunning expensive steps.

In [ ]:
# Example reload commands:
# adata = sc.read_h5ad(OUT / f"{CONFIG['project_name']}_mrvi_latents.h5ad")
# sample_normalized_dists = xr.open_dataset(OUT / f"{CONFIG['project_name']}_sample_normalized_dists.nc")
# sample_dist_df = pd.read_csv(OUT / f"{CONFIG['project_name']}_sample_distance_matrix.csv", index_col=0)
# sample_cluster = pd.read_csv(OUT / f"{CONFIG['project_name']}_sample_response_clusters.csv", index_col=0)
# model = MRVI.load(OUT / CONFIG["model_dir"], adata=adata)
# counterfactual_chunk_paths = pd.read_csv(OUT / f"{CONFIG['project_name']}_counterfactual_distance_chunks.csv")["path"].tolist()
